In [ ]:
%sql

--Cria catalog e schema
CREATE CATALOG IF NOT EXISTS lh_nautical;
CREATE SCHEMA IF NOT EXISTS lh_nautical.bronze;

In [ ]:
from grupo10.utils.logger import get_logger
from grupo10.utils.audit import ensure_audit_table, write_audit_record
#Databricks ja importa spark globalmente, mas pylance no VSC grita se não importar deliberadamente.
from databricks.sdk.runtime import spark
from datetime import datetime  # EVA — necessário para o batch_id

#TODO definir a a string do nome da tabela de audit em outro local e referenciar aqui?
#O underline antes do nome da tabela de auditoria de ingestão é uma convenção para dizer que ela é interna, ou seja, não é uma tabela do negócio em si
AUDIT_TABLE = "lh_nautical.bronze._ingestion_audit"

logger = get_logger("sqlserver_ingest")
ensure_audit_table(spark, AUDIT_TABLE)
logger.info(f"Audit table ready: {AUDIT_TABLE}")


In [ ]:
# Isolando a infraestrutura de conexão
from databricks.sdk.runtime import dbutils

db_user = dbutils.secrets.get(scope="sqlserver", key="username")
db_password = dbutils.secrets.get(scope="sqlserver", key="password")
db_host = dbutils.secrets.get(scope="sqlserver", key="host")
db_port = dbutils.secrets.get(scope="sqlserver", key="port")
db_database = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{db_host}:{db_port};databaseName={db_database}"

# EVA — credenciais agrupadas num dict para passar às funções de ingestao.
# Quem chama é responsável por fornecer as credenciais -> funções puras e testáveis.
conn = {"user": db_user, "password": db_password}

# EVA — Identificador único do lote para garantir rastreabilidade.
# Inclui hora/min/seg para que dois runs no mesmo dia tenham _batch_id distintos;
# assim conseguimos identificar e reprocessar exatamente o lote problemático.
# Gerado UMA vez aqui (antes do loop) para que todas as tabelas do run
# compartilhem o MESMO _batch_id.
batch_id = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

logger.info(f"Batch ID: {batch_id}")  # EVA — loga o batch_id para rastreabilidade

In [ ]:
import json
import os

#TODO Verificar como configura parametros de jobs -> Isso aqui é um hack para achar a pasta do tabelas_config.json de dentro do Databricks. O padrão ja industria seria colocar esses dados como parametros do Job.

#Encontra o  '/Users/user@company.com/grupo_10/src/sqlserver_ingest'
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
#Remove o nome do notebook para obter o '/Users/user@company.com/grupo_10/src'
notebook_dir = os.path.dirname(notebook_path)

#Adiciona o Workspace para obtermos o '/Workspace/Users/user@company.com/grupo_10/src'
if not notebook_dir.startswith("/Workspace"):
    notebook_dir = "/Workspace" + notebook_dir

#Adiciona o tabelas config para fica '/Workspace/Users/user@company.com/grupo_10/src/tabelas_config.json'
tabelas_config_path = os.path.join(notebook_dir, "tabelas_config.json")

with open(tabelas_config_path, "r", encoding="utf-8") as f:
    target_tables = json.load(f)


In [ ]:
import time

from grupo10.ingestion import extrair_tabela, carregar_bronze
from databricks.sdk.runtime import spark  # usado por write_audit_record

# Lista das tabelas que vão ser extraídas - Geralmente é obtido através de outro banco de dados que chama de ControlDB, mas aqui no nosso projeto não vejo problema de apenas colocarmos eles num JSON dentro do databricks
# target_tables = [
    # "customers",
    # "ordersss",
#   "order_items",
#   "products",
#   "product_variants",
#   "brands",
#   "categories",
#   "returns",
#   "return_items",
#   "stock_levels",
#   "stock_movements",
#   "suppliers",
#   "employees",
#   "locatiozzns",
#     "addresses",
# ]

# Loop para processar cada tabela. A complexidade técnica (opções JDBC, colunas de
# auditoria, modo de escrita) está encapsulada em ingestion.py; aqui o notebook
# apenas orquestra: loop, logging e auditoria.
failed_tables = []

for table in target_tables:
    start = time.time()
    try:
        logger.info(f"Starting extraction: {table}")

        # Extrai do SQL Server (já com as colunas de auditoria) e grava na Bronze.
        df_temp = extrair_tabela(table, jdbc_url, conn, batch_id)
        output_table = carregar_bronze(df_temp, table)

        row_count = df_temp.count()
        duration = time.time() - start
        logger.info(f"Done: {output_table} | rows={row_count} | duration={duration:.2f}s")

        write_audit_record(
            spark=spark,
            table_name=table,
            status="success",
            duration_seconds=duration,
            row_count=row_count,
            audit_table=AUDIT_TABLE,
        )

    except Exception as e:
        duration = time.time() - start
        logger.error(f"Failed: {table} | duration={duration:.2f}s | error={e}")
        failed_tables.append(table)

        write_audit_record(
            spark=spark,
            table_name=table,
            status="failed",
            duration_seconds=duration,
            error_message=str(e),
            audit_table=AUDIT_TABLE,
        )

if failed_tables:
    logger.warning(f"Ingestion finished with {len(failed_tables)} failure(s): {failed_tables}")
else:
    logger.info("All tables ingested successfully.")
